# 03 · Mask-label diagnostic: what DiffSeg30k's mask values mean (Milestone 2, 14 Nov 2026)

Notebook 02 found that about half of DiffSeg30k's edited training images have a mask with **no
zero pixel** (median edited share 100%, mostly 1024×1024 images), so no 256 crop has 1–90% edited
pixels. Spec section 8 assumes the unedited background of AI-generated base images is labelled 0.
This notebook measures every DiffSeg30k mask (train and validation, all roles), read-only:

- which values cover each image and how much: full masks (no 0) vs partial ones, by size;
- for full masks: how many values, the dominant value and its share, the second value's share;
- per value 1–8: how often it is the dominant (≥ 50%) value vs a minority (edit-like) region;
- the Glide / Kolors hold-out: does value 2 or 6 occur only as a full mask's dominant value
  (background-like) or as an edit region?
- a what-if, counted only: a full mask's dominant value read as background. How many training
  images would then give a valid crop, how many have no edit left, and how roles would change.
- `runs/checkpoints/mask_examples.png`: 8 full and 4 partial masks beside their images (colours:
  0 black, 1 red, 2 green, 3 yellow, 4 blue, 5 orange, 6 purple, 7 cyan, 8 magenta).

**Result** (send to Claude): `Paper 1/runs/checkpoints/mask_diagnostic.json`; please also look at
`mask_examples.png`. CPU runtime, about 20–30 min (10 of them copying DiffSeg30k). Nothing changes
in training, roles or the spec until Dhanush decides.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
import json, logging, os, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import get_paths, session_info

WORKERS = os.cpu_count() or 2
T0 = time.time()
info = session_info()
paths = get_paths()
if DRY:   # a fresh fake Drive of its own
    import shutil, tempfile
    from pathlib import Path
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_mask_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
MANIFEST = paths.runs / 'checkpoints' / 'cp1' / 'manifest_cp1.csv.gz'
print('commit', info['git_commit'], '| workers', WORKERS, '| manifest', MANIFEST)

In [ ]:
# Cell 3: DiffSeg30k masks on the local disk (dry run: a fake world and a fake CP1 run instead)
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    W = {'sd14_per_class': 30, 'val_per_class': 2, 'diffseg': (36, 12),
         'sid_caps': {'train': 24, 'validation': 6}, 'coco_images': 60}
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', **W)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True,
                overrides=cp1_overrides(W['coco_images']))
    dry_cfg = CP1Config(workers=2, crops_per_class=150, blockiness_per_group=40, base_unclear_max=0.2,
                        diffseg_card_splits={'train': W['diffseg'][0], 'validation': W['diffseg'][1]})
    dry = run_cp1(paths, dry_cfg, out_dir=MANIFEST.parent, cache_dir=paths.cache / 'cp1', progress=None)
else:
    from eviground.data.download import stage
    staged = stage('diffseg30k', paths=paths)
    print('DiffSeg30k on the local disk:', staged)
    assert MANIFEST.exists(), f'{MANIFEST} not found: it is written by notebook 01 (CP1 run 4)'

In [ ]:
# Cell 4: measure every DiffSeg30k mask, save the record and the example sheet
import datetime
from eviground.data.mask_diagnostic import run_diagnostic

OUT = paths.runs / 'checkpoints'
diag = run_diagnostic(MANIFEST, out_dir=OUT, workers=WORKERS)
record = {
    'diagnostic': 'DiffSeg30k mask labels',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY, 'branch': BRANCH, 'session': info,
    'wall_time_h': round((time.time() - T0) / 3600, 3),
    'result': diag,
}
out = OUT / 'mask_diagnostic.json'
out.write_text(json.dumps(record, indent=1, default=str))
print('saved', out, 'and', OUT / diag['examples']['file'], '| wall time',
      round((time.time() - T0) / 60, 1), 'min')
print('masks:', diag['masks'], '| empty:', diag['empty_masks'])
print('full masks:', diag['full_masks'])
print('partial masks:', diag['partial_masks'])
print('by role | size | full:')
for k, v in diag['by_role_size_full'].items():
    print(f'  {k}: {v}')
print('per value, dominant (>= 50%) vs minority, by size:')
for k, v in diag['per_value_dominant_vs_minority'].items():
    print(f'  {k}: {v}')
print('hold-out values:', diag['holdout_values'])
print('what-if (dominant value of a full mask = background):', diag['whatif_dominant_as_background'])